# 02 — Análise Exploratória (EDA)

**Objetivo:** responder às hipóteses de `docs/business_problem.md` a partir dos dados limpos de `data/processed/` (gerados por `src/clean_data.py`).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True

PROCESSED = Path('../data/processed')

orders = pd.read_parquet(PROCESSED / 'orders_clean.parquet')
order_items = pd.read_parquet(PROCESSED / 'order_items_clean.parquet')
payments = pd.read_parquet(PROCESSED / 'payments_clean.parquet')
print(f'{orders["order_id"].nunique()} pedidos limpos')

## 1. Evolução mensal de pedidos e receita (H2/KPIs)

In [ ]:
monthly = (
    orders
    .groupby('order_month')
    .agg(pedidos=('order_id', 'nunique'), receita=('order_revenue', 'sum'))
)
monthly['ticket_medio'] = monthly['receita'] / monthly['pedidos']

fig, ax = plt.subplots()
ax.bar(monthly.index.astype(str), monthly['pedidos'], label='Pedidos')
ax2 = ax.twinx()
ax2.plot(monthly.index.astype(str), monthly['receita'], color='tab:red', label='Receita')
ax.set_title('Pedidos e receita por mês')
ax.tick_params(axis='x', rotation=60)
fig.legend(loc='upper left')
plt.show()

monthly.tail(12)

## 2. Atraso de entrega × nota de avaliação (H1)

In [ ]:
delay_score = (
    orders.dropna(subset=['delivery_delay_days'])
    .assign(delay_bucket=pd.cut(
        orders['delivery_delay_days'],
        bins=[-999, -10, -3, 0, 3, 10, 999],
        labels=['>10d adiantado', '4-10d adiantado', 'até 3d adiantado', 'até 3d atrasado', '4-10d atrasado', '>10d atrasado'],
    ))
    .groupby('delay_bucket', observed=True)
    .agg(nota_media=('review_score', 'mean'), pedidos=('order_id', 'nunique'))
)
delay_score.plot(kind='bar', y='nota_media', legend=False, title='Nota média por faixa de atraso')
plt.show()
delay_score

## 3. Receita por categoria e por estado (H2)

In [ ]:
cat_rev = (
    order_items
    .groupby('product_category_name_english')['order_revenue']
    .sum().sort_values(ascending=False)
)
top10 = cat_rev.head(10)
pareto = cat_rev.cumsum() / cat_rev.sum()
n80 = (pareto <= 0.8).sum()
print(f'{n80} de {len(cat_rev)} categorias respondem por 80% da receita')

top10.sort_values().plot(kind='barh', title='Top 10 categorias por receita')
plt.show()

state_rev = orders.groupby('customer_state').agg(
    receita=('order_revenue', 'sum'), pedidos=('order_id', 'nunique'),
).sort_values('receita', ascending=False)
state_rev.head(10)

## 4. Meio de pagamento, parcelas e ticket (H4)

In [ ]:
pay = (
    payments
    .groupby('payment_type')
    .agg(pedidos=('order_id', 'nunique'), valor_total=('payment_value', 'sum'))
    .sort_values('valor_total', ascending=False)
)
pay['ticket_medio'] = pay['valor_total'] / pay['pedidos']
pay

## 5. Frete × satisfação (H5)

In [ ]:
orders.plot(
    kind='scatter', x='freight_ratio', y='review_score',
    alpha=0.1, title='Proporção do frete no pedido × nota',
)
plt.show()

orders.assign(frete_bucket=pd.qcut(orders['freight_ratio'], 5, duplicates='drop')) \
      .groupby('frete_bucket', observed=True)['review_score'].mean()

## Conclusões

- a preencher com os achados por hipótese (H1–H5)
- alimentar o dashboard Power BI com as views de `sql/04_views.sql`